# Full Experiment: Fourier Signal Classification
## Run on Google Colab with Free GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbass12/FourierSeriesClassification/blob/main/notebooks/Full_Experiment_Colab.ipynb)

This notebook runs the complete experiment suite with larger dataset sizes using Colab's free GPU.

In [ ]:
# Clone the repository
!git clone https://github.com/abbass12/FourierSeriesClassification.git
%cd FourierSeriesClassification
!pip install -q torch numpy scipy matplotlib seaborn

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from signals import generate_dataset, generate_grid, train_test_split_signals, SIGNAL_NAMES
from fourier import signals_to_fourier_features, signals_to_fourier_with_jumps
from models import (SignalClassifier, SignalClassifierWithJumps,
                    train_model, evaluate_model,
                    prepare_dataloader, prepare_dataloader_with_jumps)
import numpy as np
import json
import os

# Full experiment parameters (larger than sandbox run)
n_samples = 2000  # per signal type
n_points = 1500
x = generate_grid(n_points)
seed = 42
batch_size = 64
n_epochs = 80
results = {}

## Model A: Raw Signal Data

In [ ]:
print('=== MODEL A: Raw Signal Data ===')
snr_levels = [None, 30, 25, 20, 15]
model_a_results = {}
for snr in snr_levels:
    label = f'SNR_{snr}dB' if snr else 'clean'
    print(f'\n--- {label} ---')
    X, y = generate_dataset(n_samples, n_points, snr_db=snr, seed=seed)
    splits = train_test_split_signals(X, y, seed=seed)
    tl = prepare_dataloader(splits['X_train'], splits['y_train'], batch_size)
    vl = prepare_dataloader(splits['X_val'], splits['y_val'], batch_size, False)
    tel = prepare_dataloader(splits['X_test'], splits['y_test'], batch_size, False)
    model = SignalClassifier(n_points, 5)
    hist = train_model(model, tl, vl, n_epochs=n_epochs, device=device)
    acc, cm = evaluate_model(model, tel, device=device)
    model_a_results[label] = {'accuracy': float(acc), 'confusion_matrix': cm.tolist(),
                              'history': {k:[float(v) for v in vals] for k,vals in hist.items()}}
    print(f'  Test Accuracy: {acc:.4f}')
results['model_a'] = model_a_results

## Model B: Fourier Coefficients

In [ ]:
print('=== MODEL B: Fourier Coefficients ===')
X_clean, y_clean = generate_dataset(n_samples, n_points, snr_db=None, seed=seed)
splits_clean = train_test_split_signals(X_clean, y_clean, seed=seed)
model_b_results = {}
for n_modes in [10, 20, 30, 50, 75, 100, 150, 200]:
    print(f'\n--- N={n_modes} ---')
    X_tr = signals_to_fourier_features(splits_clean['X_train'], n_modes)
    X_va = signals_to_fourier_features(splits_clean['X_val'], n_modes)
    X_te = signals_to_fourier_features(splits_clean['X_test'], n_modes)
    tl = prepare_dataloader(X_tr, splits_clean['y_train'], batch_size)
    vl = prepare_dataloader(X_va, splits_clean['y_val'], batch_size, False)
    tel = prepare_dataloader(X_te, splits_clean['y_test'], batch_size, False)
    model = SignalClassifier(2*n_modes, 5)
    hist = train_model(model, tl, vl, n_epochs=n_epochs, device=device)
    acc, cm = evaluate_model(model, tel, device=device)
    model_b_results[f'N_{n_modes}'] = {'accuracy': float(acc), 'confusion_matrix': cm.tolist(),
                                        'history': {k:[float(v) for v in vals] for k,vals in hist.items()}}
    print(f'  Test Accuracy: {acc:.4f}')
results['model_b'] = model_b_results

## Model C: Fourier + Jump Features

In [ ]:
print('=== MODEL C: Fourier + Jump Features ===')
model_c_results = {}
max_jumps = 4
for n_modes in [10, 20, 30, 50, 75, 100, 150, 200]:
    print(f'\n--- N={n_modes} (with jumps) ---')
    X_tr = signals_to_fourier_with_jumps(splits_clean['X_train'], x, n_modes, max_jumps=max_jumps)
    X_va = signals_to_fourier_with_jumps(splits_clean['X_val'], x, n_modes, max_jumps=max_jumps)
    X_te = signals_to_fourier_with_jumps(splits_clean['X_test'], x, n_modes, max_jumps=max_jumps)
    fd = 2*n_modes
    jd = 2*max_jumps
    tl = prepare_dataloader_with_jumps(X_tr[:,:fd], X_tr[:,fd:], splits_clean['y_train'], batch_size)
    vl = prepare_dataloader_with_jumps(X_va[:,:fd], X_va[:,fd:], splits_clean['y_val'], batch_size, False)
    tel = prepare_dataloader_with_jumps(X_te[:,:fd], X_te[:,fd:], splits_clean['y_test'], batch_size, False)
    model = SignalClassifierWithJumps(fd, jd, 5)
    hist = train_model(model, tl, vl, n_epochs=n_epochs, device=device, model_type='C')
    acc, cm = evaluate_model(model, tel, device=device, model_type='C')
    model_c_results[f'N_{n_modes}'] = {'accuracy': float(acc), 'confusion_matrix': cm.tolist(),
                                        'history': {k:[float(v) for v in vals] for k,vals in hist.items()}}
    print(f'  Test Accuracy: {acc:.4f}')
results['model_c'] = model_c_results

## Save Results and Generate Figures

In [ ]:
os.makedirs('results', exist_ok=True)
with open('results/experiment_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f"Model A (clean):  {results['model_a']['clean']['accuracy']:.4f}")
print(f"Model B (N=50):   {results['model_b']['N_50']['accuracy']:.4f}")
print(f"Model C (N=50):   {results['model_c']['N_50']['accuracy']:.4f}")
print(f"Model C (N=100):  {results['model_c']['N_100']['accuracy']:.4f}")
print(f"Model C (N=200):  {results['model_c']['N_200']['accuracy']:.4f}")

In [ ]:
from plotting import generate_all_paper_figures
generate_all_paper_figures(results, 'results/figures')
print('All figures saved!')